# Visualize VoxBind Dataset

Explore the CrossDocked dataset used by VoxBind, select one pocket–ligand pair, and visualize it.

In [1]:
import sys, os
import time

# make voxbind importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import torch
import numpy as np
import plotly.graph_objects as go

from voxbind.constants import ATOM_ELEMENTS, ELEMENTS_HASH_CROSSDOCKED

## 1. Load and explore the dataset

In [2]:
DATA_DIR = os.path.join(os.getcwd(), "..", "voxbind", "dataset", "data")

# Choose which split to visualize
SPLIT = "test"   # "test" or "train"

# Load requested split (train is larger; test is smaller/faster)
pt_name = f"data_{SPLIT}.pt"
data_split = torch.load(os.path.join(DATA_DIR, pt_name), weights_only=False)

print(f"Loaded split               : {SPLIT}")
print(f"Number of samples in split : {len(data_split)}")
print(f"Type of each sample        : {type(data_split[0])}")
print(f"Each sample is a tuple of  : (pocket_dict, ligand_dict)\n")

pocket_sample, ligand_sample = data_split[0]

print("=== Pocket dict keys ===")
for k, v in pocket_sample.items():
    info = f"  shape={v.shape}, dtype={v.dtype}" if isinstance(v, torch.Tensor) else f"  {v}"
    print(f"  {k:20s} {info}")

print("\n=== Ligand dict keys ===")
for k, v in ligand_sample.items():
    info = f"  shape={v.shape}, dtype={v.dtype}" if isinstance(v, torch.Tensor) else f"  {v}"
    print(f"  {k:20s} {info}")

Loaded split               : test
Number of samples in split : 100
Type of each sample        : <class 'tuple'>
Each sample is a tuple of  : (pocket_dict, ligand_dict)

=== Pocket dict keys ===
  id                     BSD_ASPTE_1_130_0/2z3h_A_rec_1wn6_bst_lig_tt_docked_3_pocket10.pdb
  coords                 shape=torch.Size([409, 3]), dtype=torch.float32
  atoms_channel          shape=torch.Size([409]), dtype=torch.uint8

=== Ligand dict keys ===
  id                     BSD_ASPTE_1_130_0/2z3h_A_rec_1wn6_bst_lig_tt_docked_3.sdf
  coords                 shape=torch.Size([31, 3]), dtype=torch.float32
  atoms_channel          shape=torch.Size([31]), dtype=torch.uint8
  max_len                16.34


## 2. Select one sample and inspect it

In [4]:
# Select sample by PDB ID (4-letter code from the pocket id, e.g. "2Z3H")
TARGET_PDB_ID = "14gs"  # change this to the PDB ID you want to inspect
SEARCH_OTHER_SPLIT_IF_MISSING = True   # if not found in current SPLIT, try the other split
SEARCH_ALL_PT_FILES_IF_MISSING = False # last-resort: scan any data_*.pt in DATA_DIR


def extract_pdb_id(pocket_id: str) -> str:
    """Extract 4-letter PDB ID from a CrossDocked pocket id string."""
    return pocket_id.split("/")[1].split("_")[0].upper()


def find_sample_by_pdb_id(pdb_id: str, data):
    """Return (idx, (pocket_dict, ligand_dict)) for the first matching sample, else None."""
    pdb_id = pdb_id.upper()
    for i, (poc, lig) in enumerate(data):
        if extract_pdb_id(poc["id"]) == pdb_id:
            return i, (poc, lig)
    return None


# 1) Search within the currently loaded split (data_split)
found = find_sample_by_pdb_id(TARGET_PDB_ID, data_split)
source_pt = f"data_{SPLIT}.pt"
source_split = SPLIT

# 2) If missing, optionally try the other split (train <-> test)
if found is None and SEARCH_OTHER_SPLIT_IF_MISSING:
    other_split = "train" if SPLIT == "test" else "test"
    other_pt = os.path.join(DATA_DIR, f"data_{other_split}.pt")
    try:
        other_data = torch.load(other_pt, weights_only=False)
        found = find_sample_by_pdb_id(TARGET_PDB_ID, other_data)
        if found is not None:
            source_pt = f"data_{other_split}.pt"
            source_split = other_split
            data_split = other_data  # switch to the split that actually contains the PDB
            SPLIT = other_split
    except Exception as e:
        print(f"Warning: failed to load other split {other_split}: {e}")

# 3) Last resort: scan other data_*.pt files
if found is None and SEARCH_ALL_PT_FILES_IF_MISSING:
    from pathlib import Path

    data_dir = Path(DATA_DIR)
    for pt_path in sorted(data_dir.glob("data_*.pt")):
        try:
            data_any = torch.load(pt_path, weights_only=False)
        except Exception as e:
            print(f"Warning: failed to load {pt_path.name}: {e}")
            continue
        found = find_sample_by_pdb_id(TARGET_PDB_ID, data_any)
        if found is not None:
            source_pt = pt_path.name
            source_split = pt_path.stem.replace("data_", "")
            data_split = data_any
            SPLIT = source_split
            break

if found is None:
    raise ValueError(
        f"PDB ID {TARGET_PDB_ID.upper()!r} was not found in the VoxBind dataset files under {DATA_DIR}."
    )

SAMPLE_IDX, (pocket_raw, ligand_raw) = found

print(f"Found PDB ID {TARGET_PDB_ID.upper()} in {source_pt} (SPLIT={source_split}) at SAMPLE_IDX={SAMPLE_IDX}")

# reverse lookup: channel index -> element name
IDX_TO_ELEM = {v: k for k, v in ELEMENTS_HASH_CROSSDOCKED.items()}

print(f"Pocket id  : {pocket_raw['id']}")
print(f"  #atoms   : {pocket_raw['coords'].shape[0]}")
print(f"  elements : {dict(zip(*np.unique(pocket_raw['atoms_channel'].numpy(), return_counts=True)))}")
print()
print(f"Ligand id  : {ligand_raw['id']}")
print(f"  #atoms   : {ligand_raw['coords'].shape[0]}")
print(f"  max_len  : {ligand_raw['max_len']} Å")
elem_counts = {}
for ch in ligand_raw["atoms_channel"].numpy():
    elem_counts[IDX_TO_ELEM.get(ch, f"ch{ch}")] = elem_counts.get(IDX_TO_ELEM.get(ch, f"ch{ch}"), 0) + 1
print(f"  elements : {elem_counts}")

Found PDB ID 14GS in data_test.pt (SPLIT=test) at SAMPLE_IDX=3
Pocket id  : GSTP1_HUMAN_2_210_0/14gs_A_rec_20gs_cbd_lig_tt_min_0_pocket10.pdb
  #atoms   : 255
  elements : {np.uint8(0): np.int64(161), np.uint8(1): np.int64(47), np.uint8(2): np.int64(45), np.uint8(3): np.int64(2)}

Ligand id  : GSTP1_HUMAN_2_210_0/14gs_A_rec_20gs_cbd_lig_tt_min_0.sdf
  #atoms   : 22
  max_len  : 7.86 Å
  elements : {'C': 14, 'N': 2, 'O': 5, 'S': 1}


## 3. Visualize raw 3D atom coordinates

Interactive 3D scatter plot showing the pocket (grey) and ligand atoms (coloured by element).

In [8]:
ELEM_COLORS = {
    "C": "gray", "O": "red", "N": "blue", "S": "gold",
    "F": "green", "Cl": "lime", "P": "orange", "H": "white",
}

fig = go.Figure()

# --- Pocket atoms (all shown in light grey) ---
poc_coords = pocket_raw["coords"].numpy()
fig.add_trace(go.Scatter3d(
    x=poc_coords[:, 0], y=poc_coords[:, 1], z=poc_coords[:, 2],
    mode="markers",
    marker=dict(size=3, color="lightgrey", opacity=0.85),
    name="Pocket",
))

# --- Ligand atoms (coloured by element) ---
lig_coords = ligand_raw["coords"].numpy()
lig_channels = ligand_raw["atoms_channel"].numpy()

for ch_idx in np.unique(lig_channels):
    elem = IDX_TO_ELEM.get(int(ch_idx), f"ch{int(ch_idx)}")
    mask = lig_channels == ch_idx
    fig.add_trace(go.Scatter3d(
        x=lig_coords[mask, 0], y=lig_coords[mask, 1], z=lig_coords[mask, 2],
        mode="markers",
        marker=dict(size=5, color=ELEM_COLORS.get(elem, "purple")),
        name=f"Ligand {elem}",
    ))

fig.update_layout(
    title=f"Pocket + Ligand  —  {pocket_raw['id']}",
    scene=dict(
        xaxis_title="x (Å)", yaxis_title="y (Å)", zaxis_title="z (Å)",
        aspectmode="data",
    ),
    width=800, height=650,
    legend=dict(itemsizing="constant"),
)
fig.show()

## 4. Original vs Voxelized — side-by-side comparison

Load the sample through `DatasetCrossdocked`, voxelize it, and compare the original atom coordinates with the voxelized representation for **pocket only**, **ligand only**, and **both**.

In [6]:
from plotly.subplots import make_subplots
from voxbind.dataset.crossdocked import DatasetCrossdocked
from voxbind.voxelizer import Voxelizer

# ── Build dataset & voxelizer ────────────────────────────────────────
# Use the same SPLIT as the sample-selection section above.
dset = DatasetCrossdocked(
    data_dir=DATA_DIR, split=SPLIT, aug=False,
    small=True, max_len=30, verbose=True,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
voxelizer = Voxelizer(grid_dim=64, device=device)

# ── Pick sample by PDB ID (same SAMPLE_IDX as above), voxelize ───────
sample = dset[SAMPLE_IDX]

poc = sample["pocket"]
lig = sample["ligand"]

pocket_batch = {k: v.unsqueeze(0) for k, v in poc.items() if isinstance(v, torch.Tensor)}
ligand_batch = {k: v.unsqueeze(0) for k, v in lig.items() if isinstance(v, torch.Tensor)}

vox_pocket = voxelizer(pocket_batch, num_channels=4)[0].cpu()   # [4,64,64,64]
vox_ligand = voxelizer(ligand_batch, num_channels=7)[0].cpu()   # [7,64,64,64]

# ── Original coords (remove 999 padding) ────────────────────────────
poc_mask = poc["atoms_channel"] != 999
orig_poc_coords   = poc["coords"][poc_mask].numpy()
orig_poc_channels = poc["atoms_channel"][poc_mask].numpy().astype(int)

lig_mask = lig["atoms_channel"] != 999
orig_lig_coords   = lig["coords"][lig_mask].numpy()
orig_lig_channels = lig["atoms_channel"][lig_mask].numpy().astype(int)

# ── Helper: voxel grid → scatter points ──────────────────────────────
def voxel_to_scatter(vox, threshold=0.1, resolution=0.25):
    """Convert [C, D, D, D] voxel grid to (coords_np, channels_np)."""
    grid_dim = vox.shape[-1]
    center = (grid_dim - 1) / 2
    all_coords, all_ch = [], []
    for ch in range(vox.shape[0]):
        ix, iy, iz = torch.where(vox[ch] > threshold)
        if len(ix) == 0:
            continue
        x = (ix.float() - center) * resolution
        y = (iy.float() - center) * resolution
        z = (iz.float() - center) * resolution
        all_coords.append(torch.stack([x, y, z], dim=1))
        all_ch.append(torch.full((len(ix),), ch, dtype=torch.long))
    if not all_coords:
        return np.empty((0, 3)), np.empty(0, dtype=int)
    return torch.cat(all_coords).numpy(), torch.cat(all_ch).numpy()

vox_poc_coords, vox_poc_ch = voxel_to_scatter(vox_pocket, threshold=0.05)
vox_lig_coords, vox_lig_ch = voxel_to_scatter(vox_ligand, threshold=0.05)

print(f"Original  pocket atoms : {orig_poc_coords.shape[0]},  ligand atoms : {orig_lig_coords.shape[0]}")
print(f"Voxelized pocket pts   : {vox_poc_coords.shape[0]},  ligand pts   : {vox_lig_coords.shape[0]}")

| filter by size reduce n ligands from 100 to 100
Original  pocket atoms : 255,  ligand atoms : 22
Voxelized pocket pts   : 233382,  ligand pts   : 3103


In [7]:
# ── Helper: add atom scatter traces to a figure ─────────────────────

POCKET_ELEM_COLORS = {0: "silver", 1: "salmon", 2: "cornflowerblue", 3: "khaki"}
LIGAND_ELEM_COLORS = {0: "gray", 1: "red", 2: "blue", 3: "gold", 4: "green", 5: "lime", 6: "orange"}

def add_atom_traces(fig, coords, channels, elem_colors, name_prefix, row, col,
                    size=3, opacity=0.5):
    for ch_idx in np.unique(channels):
        elem = IDX_TO_ELEM.get(int(ch_idx), f"ch{int(ch_idx)}")
        mask = channels == ch_idx
        fig.add_trace(go.Scatter3d(
            x=coords[mask, 0], y=coords[mask, 1], z=coords[mask, 2],
            mode="markers",
            marker=dict(size=size, color=elem_colors.get(int(ch_idx), "purple"), opacity=opacity),
            name=f"{name_prefix} {elem}",
            legendgroup=f"{name_prefix}_{elem}",
        ), row=row, col=col)

def style_scenes(fig, **kwargs):
    axis = dict(title="", showbackground=False, showticklabels=False)
    for key in [k for k in fig.layout.to_plotly_json() if k.startswith("scene")]:
        fig.layout[key].update(
            xaxis=axis, yaxis=axis, zaxis=axis,
            aspectmode="data", **kwargs,
        )

In [ ]:
# ── 4a. Pocket only — Original vs Voxelized ───────────────────────────
fig_poc = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Pocket — Original", "Pocket — Voxelized"),
)

add_atom_traces(fig_poc, orig_poc_coords, orig_poc_channels, POCKET_ELEM_COLORS,
                "Poc", row=1, col=1, size=3, opacity=0.5)
add_atom_traces(fig_poc, vox_poc_coords, vox_poc_ch, POCKET_ELEM_COLORS,
                "Vox-Poc", row=1, col=2, size=2, opacity=0.35)

style_scenes(fig_poc)
fig_poc.update_layout(title_text="Pocket Only", width=1100, height=550,
                      legend=dict(itemsizing="constant"))
fig_poc.show()

In [ ]:
# ── 4a-surface. Pocket surface — Original (alpha-hull) vs Voxelized (isosurface)
fig_surf = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Pocket Surface — Original", "Pocket Surface — Voxelized"),
)

# ── Left: original atoms + translucent alpha-hull surface ────────────
add_atom_traces(fig_surf, orig_poc_coords, orig_poc_channels, POCKET_ELEM_COLORS,
                "Poc", row=1, col=1, size=2, opacity=0.4)
fig_surf.add_trace(go.Mesh3d(
    x=orig_poc_coords[:, 0],
    y=orig_poc_coords[:, 1],
    z=orig_poc_coords[:, 2],
    alphahull=2.5,
    color="lightblue",
    opacity=0.25,
    name="Surface (alpha-hull)",
    showlegend=True,
), row=1, col=1)

# ── Right: voxelized isosurface from summed density ─────────────────
vox_sum = vox_pocket.sum(dim=0).numpy()          # [64,64,64]
grid_dim = vox_sum.shape[0]
center = (grid_dim - 1) / 2
resolution = 0.25
lin = (np.arange(grid_dim) - center) * resolution
X, Y, Z = np.meshgrid(lin, lin, lin, indexing="ij")

fig_surf.add_trace(go.Isosurface(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=vox_sum.flatten(),
    isomin=0.08,
    isomax=0.20,
    opacity=0.3,
    surface_count=3,
    colorscale="Blues",
    showscale=False,
    caps=dict(x_show=False, y_show=False, z_show=False),
    name="Isosurface",
    showlegend=True,
), row=1, col=2)

style_scenes(fig_surf)
fig_surf.update_layout(
    title_text="Pocket Surface — Original vs Voxelized",
    width=1100, height=550,
    legend=dict(itemsizing="constant"),
)
fig_surf.show()

## 4a-nglview. Pocket surface with NGLView

Molecular-grade visualization using **nglview**: pocket shown as translucent surface coloured by element, ligand as ball-and-stick.

In [ ]:
import nglview as nv

crossdocked_dir = os.path.join(DATA_DIR, "crossdocked_pocket10")
pocket_pdb_path = os.path.join(crossdocked_dir, pocket_raw["id"])
ligand_sdf_path = os.path.join(crossdocked_dir, ligand_raw["id"])
print(f"Pocket PDB : {pocket_pdb_path}")
print(f"Ligand SDF : {ligand_sdf_path}")

view = nv.NGLWidget(height="600px", width="600px")

# pocket (PDB)
# view.add_component(pocket_pdb_path)
# view.clear_representations(component=0)
# view.add_representation("surface", component=0, opacity=0.8, color="electrostatic", surfaceType="av")

# ligand (SDF)
view.add_component(ligand_sdf_path)
view.clear_representations(component=1)
view.add_representation("ball+stick", component=1, colorScheme="element", multipleBond="symmetric")
view.add_representation("surface", component=1, opacity=0.2, color="white", surfaceType="vws")

view.center()
view

In [ ]:
view.download_image(
	filename="ligand.png",
	factor=4,
	transparent=True,
	antialias=True,
)

In [ ]:
# ── 4b. Ligand only — Original vs Voxelized ───────────────────────────
fig_lig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Ligand — Original", "Ligand — Voxelized"),
)

add_atom_traces(fig_lig, orig_lig_coords, orig_lig_channels, LIGAND_ELEM_COLORS,
                "Lig", row=1, col=1, size=5, opacity=0.8)
add_atom_traces(fig_lig, vox_lig_coords, vox_lig_ch, LIGAND_ELEM_COLORS,
                "Vox-Lig", row=1, col=2, size=3, opacity=0.6)

style_scenes(fig_lig)
fig_lig.update_layout(title_text="Ligand Only", width=1100, height=550,
                      legend=dict(itemsizing="constant"))
fig_lig.show()

In [ ]:
# ── 4c. Pocket + Ligand — Original vs Voxelized ───────────────────────
fig_both = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("Pocket + Ligand — Original", "Pocket + Ligand — Voxelized"),
)

# Original side
add_atom_traces(fig_both, orig_poc_coords, orig_poc_channels, POCKET_ELEM_COLORS,
                "Poc", row=1, col=1, size=3, opacity=0.3)
add_atom_traces(fig_both, orig_lig_coords, orig_lig_channels, LIGAND_ELEM_COLORS,
                "Lig", row=1, col=1, size=5, opacity=0.8)

# Voxelized side
add_atom_traces(fig_both, vox_poc_coords, vox_poc_ch, POCKET_ELEM_COLORS,
                "Vox-Poc", row=1, col=2, size=2, opacity=0.2)
add_atom_traces(fig_both, vox_lig_coords, vox_lig_ch, LIGAND_ELEM_COLORS,
                "Vox-Lig", row=1, col=2, size=3, opacity=0.6)

style_scenes(fig_both)
fig_both.update_layout(title_text="Pocket + Ligand", width=1100, height=550,
                       legend=dict(itemsizing="constant"))
fig_both.show()